# Feature Demo: Publish Pipeline (API + CLI)

This notebook demonstrates publishing workflows through both interfaces:

- Python API (`publish_record`, `publish_batch`)
- CLI (`battinfo publish ...`)

It also keeps low-level maintenance checks (`lint`, `validation`, `mapping`) visible.


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
from datetime import datetime, timezone

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
assert (ROOT / 'src').exists(), f'Repo root not found from {Path.cwd()}'
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from battinfo.api import publish_record as api_publish_record, publish_batch as api_publish_batch
from battinfo.cli import app
from typer.testing import CliRunner

runner = CliRunner()
print('ROOT:', ROOT)


## 1) Maintenance Checks

In [ ]:
def run_subprocess(cmd):
    res = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    return {
        'cmd': ' '.join(cmd),
        'exit_code': res.returncode,
        'stdout': res.stdout.strip(),
        'stderr': res.stderr.strip(),
    }

checks = [
    run_subprocess([sys.executable, 'scripts/lint_identifier_policy.py']),
    run_subprocess([sys.executable, 'scripts/validate_cells_clean.py']),
]
for c in checks:
    print('---')
    print('CMD:', c['cmd'])
    print('EXIT:', c['exit_code'])
    print('STDOUT:', c['stdout'][:300])
    if c['stderr']:
        print('STDERR:', c['stderr'][:300])


## 2) Publish One Record via API and CLI

In [ ]:
record = ROOT / 'assets' / 'examples' / 'cell-instances' / 'cell-3m6k-9t2p-7x4h-9nq8.json'
api_root = ROOT / 'tmp_publish_api'
cli_root = ROOT / 'tmp_publish_cli'
for p in (api_root, cli_root):
    if p.exists():
        shutil.rmtree(p)

api_result = api_publish_record(record, target_root=api_root)
print('API status:', api_result['status'])
print('API out:', api_result['output_dir'])

cli_res = runner.invoke(
    app,
    [
        'publish',
        'record',
        '--input',
        str(record),
        '--target-root',
        str(cli_root),
        '--format',
        'json',
    ],
)
assert cli_res.exit_code == 0, cli_res.stdout
cli_result = json.loads(cli_res.stdout)
print('CLI status:', cli_result['status'])
print('CLI out:', cli_result['output_dir'])

print('record parity status:', api_result['status'] == cli_result['status'])


## 3) Publish Batch via API and CLI

In [ ]:
source_dirs = [
    ROOT / 'assets' / 'examples' / 'cell-types',
    ROOT / 'assets' / 'examples' / 'cell-instances',
    ROOT / 'assets' / 'examples' / 'datasets',
]

api_batch = api_publish_batch(source_dirs=source_dirs, target_root=api_root)
print('API batch:', {k: api_batch[k] for k in ('status', 'processed', 'published', 'failed')})

cli_res = runner.invoke(
    app,
    [
        'publish',
        'batch',
        '--source-dir',
        str(source_dirs[0]),
        '--source-dir',
        str(source_dirs[1]),
        '--source-dir',
        str(source_dirs[2]),
        '--target-root',
        str(cli_root),
        '--format',
        'json',
    ],
)
assert cli_res.exit_code == 0, cli_res.stdout
cli_batch = json.loads(cli_res.stdout)
print('CLI batch:', {k: cli_batch[k] for k in ('status', 'processed', 'published', 'failed')})

print('batch processed parity:', api_batch['processed'] == cli_batch['processed'])


## 4) Mapping Check (CLI)

In [ ]:
mapped_out = ROOT / 'tmp_publish_domain.jsonld'
res = runner.invoke(
    app,
    [
        'map',
        str(ROOT / 'src' / 'battinfo' / 'data' / 'examples' / 'cells' / 'A123_20AH.curated.json'),
        '--target',
        'domain-battery',
        '--out',
        str(mapped_out),
    ],
)
assert res.exit_code == 0, res.stdout
mapped = json.loads(mapped_out.read_text(encoding='utf-8'))
print('mapped @type:', mapped.get('@type'))


In [ ]:
summary = {
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'checks_ok': all(c['exit_code'] == 0 for c in checks),
    'api_batch': {k: api_batch[k] for k in ('status', 'processed', 'published', 'failed')},
    'cli_batch': {k: cli_batch[k] for k in ('status', 'processed', 'published', 'failed')},
}
print(json.dumps(summary, indent=2))


In [ ]:
# Cleanup temp files/dirs (comment out if you want to inspect artifacts)
if mapped_out.exists():
    mapped_out.unlink()
for p in (api_root, cli_root):
    if p.exists():
        shutil.rmtree(p)
print('cleaned')


## Debug tips

- Keep CLI calls in `--format json` mode for machine-readable parity checks.
- If API and CLI diverge, test API first, then inspect CLI option parsing.
- Use batch summaries (`processed`, `published`, `failed`) as release-gate checks.
